# 8.20 Coreference resolution

Coreference resolution is the memory problem of NLP: decide which mentions are different names for the same entity.

Coreference begins after text has candidate mentions and asks which ones belong to the same real-world entity. Pairwise scoring estimates antecedents, then clustering and agreement constraints turn pair decisions into entity chains.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def coref_ladder():
    return [
        {"name": "D1 lesson Alice/she/Bob", "mentions": ["Alice", "she", "Bob"], "features": {(0, 1): [1, 1, 0], (0, 2): [0, 0, 0], (1, 2): [0, 0, 0]}, "mask": {(0, 1): 1, (0, 2): 1, (1, 2): 1}, "gold": {(0, 1)}, "difficulty": 1},
        {"name": "D2 clean pronoun cases", "mentions": ["Mina", "she", "Ravi", "he"], "features": {(0, 1): [1, 1, 0], (0, 2): [0, 0, 0], (0, 3): [0, 0, 0], (1, 2): [0, 0, 0], (1, 3): [0, 0, 0], (2, 3): [1, 1, 0]}, "mask": {(0, 1): 1, (0, 2): 0, (0, 3): 0, (1, 2): 0, (1, 3): 0, (2, 3): 1}, "gold": {(0, 1), (2, 3)}, "difficulty": 2},
        {"name": "D3 agreement conflicts/transitive traps", "mentions": ["Alice", "she", "the team", "it", "Bob"], "features": {(0, 1): [1, 1, 0], (0, 2): [0, 0, 0], (0, 3): [0, 0, 0], (0, 4): [0, 0, 0], (1, 2): [0, 0, 0], (1, 3): [0, 0, 0], (1, 4): [0, 0, 0], (2, 3): [1, 1, 0], (2, 4): [0, 0, 0], (3, 4): [0, 0, 0]}, "mask": {(0, 1): 1, (0, 2): 0, (0, 3): 0, (0, 4): 0, (1, 2): 0, (1, 3): 0, (1, 4): 0, (2, 3): 1, (2, 4): 0, (3, 4): 0}, "gold": {(0, 1), (2, 3)}, "difficulty": 4},
        {"name": "D4 tiny inline mini-docs", "mentions": ["Nora", "the engineer", "she", "Sam", "him"], "features": {(0, 1): [1, 0, 1], (0, 2): [1, 1, 0], (0, 3): [0, 0, 0], (0, 4): [0, 0, 0], (1, 2): [1, 1, 0], (1, 3): [0, 0, 0], (1, 4): [0, 0, 0], (2, 3): [0, 0, 0], (2, 4): [0, 0, 0], (3, 4): [1, 1, 0]}, "mask": {(0, 1): 1, (0, 2): 1, (0, 3): 0, (0, 4): 0, (1, 2): 1, (1, 3): 0, (1, 4): 0, (2, 3): 0, (2, 4): 0, (3, 4): 1}, "gold": {(0, 1), (0, 2), (3, 4)}, "difficulty": 6},
        {"name": "D5 longer distant mentions", "mentions": ["Dr Lee", "the surgeon", "patients", "they", "Lee", "she", "the clinic"], "features": {(0, 1): [1, 0, 1], (0, 2): [0, 0, 0], (0, 3): [0, 0, 0], (0, 4): [1, 0, 1], (0, 5): [1, 1, 0], (0, 6): [0, 0, 0], (1, 2): [0, 0, 0], (1, 3): [0, 0, 0], (1, 4): [1, 0, 1], (1, 5): [1, 1, 0], (1, 6): [0, 0, 0], (2, 3): [1, 1, 0], (2, 4): [0, 0, 0], (2, 5): [0, 0, 0], (2, 6): [0, 0, 0], (3, 4): [0, 0, 0], (3, 5): [0, 0, 0], (3, 6): [0, 0, 0], (4, 5): [1, 1, 0], (4, 6): [0, 0, 0], (5, 6): [0, 0, 0]}, "mask": {(0, 1): 1, (0, 2): 0, (0, 3): 0, (0, 4): 1, (0, 5): 1, (0, 6): 0, (1, 2): 0, (1, 3): 0, (1, 4): 1, (1, 5): 1, (1, 6): 0, (2, 3): 1, (2, 4): 0, (2, 5): 0, (2, 6): 0, (3, 4): 0, (3, 5): 0, (3, 6): 0, (4, 5): 1, (4, 6): 0, (5, 6): 0}, "gold": {(0, 1), (0, 4), (4, 5), (2, 3)}, "difficulty": 9},
    ]


def pairwise_f1(pred, gold):
    tp = len(pred & gold)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold) if gold else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def clusters_from_links(n_mentions, links):
    parent = list(range(n_mentions))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[rb] = ra
    for a, b in links:
        union(a, b)
    roots = {}
    labels = []
    for i in range(n_mentions):
        root = find(i)
        if root not in roots:
            roots[root] = len(roots)
        labels.append(roots[root])
    return labels


## The concept, built once (D1)\n\nA mention-pair resolver scores antecedents and masks impossible links before normalization: $$s(i,j)=w^\top\phi(m_i,m_j),\qquad p(i\mid j)=\frac{\exp(s(i,j))M_{ij}}{\sum_{k\lt j}\exp(s(k,j))M_{kj}}$$\n\nThe lesson matrix for `Alice, she, Bob` gives the `Alice→she` link score 2 and `Alice→Bob` score 0.

In [ ]:
score_matrix = np.array([[0, 2, 0], [0, 0, 0], [0, 0, 0]])
assert score_matrix[0, 1] == 2
assert score_matrix[0, 2] == 0
assert score_matrix[0, 1] - score_matrix[0, 2] == 2
assert clusters_from_links(4, {(0, 1), (2, 3)}) == [0, 0, 1, 1]
mask = np.array([[1, 1, 0], [1, 1, 0], [0, 0, 1]])
assert mask[0, 1] == 1
assert mask[0, 2] == 0
assert pairwise_f1({(0, 1), (1, 2)}, {(0, 1), (2, 3)}) == 0.5
distances = np.array([1, 5, 12])
weights = np.exp(-distances / 5)
prior = weights / weights.sum()
assert round(float(prior[0]), 6) == 0.640971
assert round(float(prior[2]), 6) == 0.071022
assert round(float(prior[0] / prior[2]), 3) == 9.025
print("Lesson numbers verified")

Now build one reusable resolver. It computes pair scores from feature vectors, applies the agreement mask before choosing links, and returns both links and clusters.

In [ ]:
def resolve_coref(mentions, features, mask, weights=np.array([1.0, 1.0, 0.5])):
    links = set()
    score_grid = np.zeros((len(mentions), len(mentions)))
    for j in range(len(mentions)):
        best = None
        for i in range(j):
            feature = np.array(features.get((i, j), [0, 0, 0]), dtype=float)
            score = float(weights @ feature)
            score_grid[i, j] = score
            if mask.get((i, j), 0) == 1:
                candidate = (score, i, j)
                if best is None or candidate > best:
                    best = candidate
        if best is not None and best[0] > 1.0:
            links.add((best[1], best[2]))
    return links, clusters_from_links(len(mentions), links), score_grid

rung = coref_ladder()[0]
links, clusters, grid = resolve_coref(rung["mentions"], rung["features"], rung["mask"])
assert links == {(0, 1)}
assert pairwise_f1(links, rung["gold"]) == 1.0
print(links, clusters)

## The dataset ladder

The ladder grows from the Alice/she lesson toy to clean pronouns, agreement conflicts, inline mini-docs, and a longer document with distant mentions.

In [ ]:
rungs = coref_ladder()
for rung in rungs:
    print(rung["name"], "mentions=", len(rung["mentions"]), "gold_links=", sorted(rung["gold"]))
    print("sample", rung["mentions"][:5])

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    links, clusters, grid = resolve_coref(rung["mentions"], rung["features"], rung["mask"])
    score = pairwise_f1(links, rung["gold"])
    rows.append((rung["name"], rung["difficulty"], score, sorted(links), clusters))
for name, difficulty, score, links, clusters in rows:
    print(f"{name:34s} difficulty={difficulty:2d} F1={score:.3f} links={links} clusters={clusters}")

## Results visualization

In [ ]:
fig, axes = plt.subplots(2, len(rungs), figsize=(16, 6))
metrics = []
difficulties = []
for col, rung in enumerate(rungs):
    links, clusters, grid = resolve_coref(rung["mentions"], rung["features"], rung["mask"])
    metrics.append(pairwise_f1(links, rung["gold"]))
    difficulties.append(rung["difficulty"])
    axes[0, col].imshow(grid, cmap="Purples")
    axes[0, col].set_title(rung["name"].split()[0])
    axes[0, col].set_xlabel("mention j")
    axes[0, col].set_ylabel("antecedent i")
    axes[1, col].bar(range(len(clusters)), clusters)
    axes[1, col].set_title("clusters")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(difficulties, metrics, marker="o")
plt.xlabel("distance / document difficulty")
plt.ylabel("pairwise F1")
plt.title("Coreference F1 vs difficulty")
plt.ylim(0, 1.05)
plt.show()

## Pitfall on the hardest rung

Surface matching and post-clustering masks are dangerous: one bad bridge can merge clusters before agreement evidence is allowed to block it.

In [ ]:
hard = rungs[-1]
surface_links = {(0, 4), (1, 4), (2, 3), (5, 6)}
post_masked = {link for link in surface_links if hard["mask"].get(link, 0) == 1}
fixed, fixed_clusters, grid = resolve_coref(hard["mentions"], hard["features"], hard["mask"])
print("surface", surface_links, "F1", pairwise_f1(surface_links, hard["gold"]))
print("post masked", post_masked, "F1", pairwise_f1(post_masked, hard["gold"]))
print("pre-link mask", fixed, "F1", pairwise_f1(fixed, hard["gold"]))
assert (5, 6) in surface_links
assert (5, 6) not in fixed

## Evaluate it + Practice

- Metric: pairwise/link F1.
- No-skill baseline: link each pronoun to the nearest previous mention.
- Cheap sanity check: agreement mask is applied before normalization.
- Ablation: remove the mask and clusters merge incompatible entities.
- Failure signals: a single bridge links two people or pairwise F1 hides bad entity chains.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.